# Módulo 08 · Aula 02 — Criando Imagens

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Fizemos o Dockerfile. Funciona. Só que a imagem tem **1,4 GB**, o build demora **6 minutos** toda vez que eu mudo uma linha de Python, e a equipe de segurança encontrou a senha do banco dentro dela."*

Três problemas, e todos têm a mesma raiz: **não entender camadas**.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Camadas | 🎯 A ideia que explica tudo |
| 2 | **Cache** | Por que a ordem das linhas importa |
| 3 | `.dockerignore` | O arquivo que quase ninguém escreve |
| 4 | Multi-stage | 1,4 GB → 180 MB |
| 5 | Escolha da base | `slim`, `alpine`, `distroless` |
| 6 | 🔴 Segredos | Por que `ARG` não esconde nada |
| 7 | 🔴 Usuário não-root | E por que `USER` importa |
| 8 | `CMD` vs `ENTRYPOINT` | E o `SIGTERM` que não chega |
| 9 | Volumes vs bind mounts | Onde os dados moram |

> 🎯 **Nesta aula você vai construir um analisador de Dockerfile.** Ele lê o arquivo, entende as camadas e aponta os problemas — antes do `docker build`. Tudo executado de verdade, sem precisar de Docker.

## ⚙️ Preparação

O mesmo modo duplo da aula anterior: análise **executada de verdade**, comandos `docker` como **referência**.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 08
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

E_LINUX = platform.system() == "Linux"
TEM_DOCKER = shutil.which("docker") is not None
DOCKER_LIGADO = False
if TEM_DOCKER:
    DOCKER_LIGADO = subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


_garantir("pyyaml", "yaml")
import yaml

print(f"sistema : {platform.system()}")
print(f"docker  : {'🐳 disponível e ligado' if DOCKER_LIGADO else ('instalado, mas o daemon não responde' if TEM_DOCKER else 'não instalado')}")


# ═══════════════════════════════════════════════════════════════
#  Executar comandos
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120) -> str:
    """Roda um comando de shell e devolve a saída."""
    processo = subprocess.run(comando, shell=True, capture_output=True,
                              text=True, cwd=cwd, timeout=timeout)
    saida = (processo.stdout + processo.stderr).rstrip()
    if mostrar and saida:
        print(saida)
    return saida


def docker(comando: str, esperado: str | None = None, mostrar: bool = True) -> str:
    """Executa `docker ...` se houver daemon; senão, mostra o comando.

    💭 Por que este modo duplo?

       Um daemon Docker não roda dentro de todo ambiente (nem dentro de
       um container, nem em CI restrito, nem neste avaliador). Em vez de
       fingir que rodou, o notebook é HONESTO: quando não há daemon, ele
       mostra o comando e a saída típica, marcada como referência.

       🔴 Saída marcada `[referência]` NÃO foi executada. Rode você
          mesmo no terminal — é assim que se aprende Docker.
    """
    linha = f"docker {comando}"
    if DOCKER_LIGADO:
        print(f"$ {linha}")
        return sh(linha, mostrar=mostrar)
    print(f"$ {linha}")
    if esperado:
        for l in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {l}")
    print("  ── [referência] o daemon Docker não está disponível aqui ──")
    return esperado or ""


# ═══════════════════════════════════════════════════════════════
#  Pasta de trabalho
# ═══════════════════════════════════════════════════════════════
def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = ""):
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        marca = "└── " if ultimo else "├── "
        tamanho = f"  ({item.stat().st_size} B)" if item.is_file() else ""
        print(f"{prefixo}{marca}{item.name}{tamanho}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]):
    """Tabela ASCII alinhada.

    ⚠️ Marcadores ASCII, não emoji: `len("⚠️")` é 2 mas o terminal
       desenha 1 coluna, e a tabela sai torta. Você já viu isso no M03.
    """
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("✅ `sh()`, `docker()`, `preparar()`, `arvore()` e `tabela()` prontos")

## 1. Camadas — a ideia que explica tudo

Uma imagem Docker não é um arquivo. É uma **pilha de camadas**, cada uma somente-leitura, cada uma guardando apenas o que **mudou** em relação à anterior.

```
┌─────────────────────────────┐  ← camada gravável (do CONTAINER)
├─────────────────────────────┤
│ COPY src/ ./src/            │  camada 5   2 MB
│ RUN pip install ...         │  camada 4  180 MB
│ COPY pyproject.toml ./      │  camada 3   1 KB
│ WORKDIR /app                │  camada 2   0 B
│ FROM python:3.12-slim       │  camada 1  130 MB
└─────────────────────────────┘
```

Três consequências que decidem tudo o mais:

| Consequência | Implicação |
|--------------|------------|
| Cada camada é **imutável** | Apagar um arquivo numa camada seguinte **não** o remove da imagem |
| Cada camada tem um **hash** | Se a entrada não mudou, o Docker reaproveita — é o cache |
| Camadas são **compartilhadas** | 10 imagens da mesma base ocupam a base uma vez só |

In [ ]:
BASE = preparar("aula_08_02")

# Instruções que criam camada — e as que não criam
cria = {"FROM": "a imagem base inteira",
        "RUN": "tudo que o comando escreveu no sistema de arquivos",
        "COPY": "os arquivos copiados",
        "ADD": "idem (e mais coisas — veja a seção 6)",
        "WORKDIR": "cria o diretório se não existir"}
nao_cria = {"ENV": "metadado", "ARG": "variável só do build",
            "EXPOSE": "documentação", "LABEL": "metadado",
            "USER": "metadado", "CMD": "metadado",
            "ENTRYPOINT": "metadado", "VOLUME": "metadado",
            "HEALTHCHECK": "metadado"}

print("CRIAM camada (ocupam espaço):")
for k, v in cria.items():
    print(f"   {k:<12} {v}")
print("\nNÃO criam camada (só metadado):")
for k, v in nao_cria.items():
    print(f"   {k:<12} {v}")

print("\n💭 Por que isso importa: 20 linhas de ENV custam zero bytes.")
print("   Uma linha de RUN mal escrita custa 300 MB.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Um leitor de Dockerfile — usado o resto da aula
# ═══════════════════════════════════════════════════════════════
import fnmatch
import re

INSTRUCOES_QUE_CRIAM_CAMADA = {"FROM", "RUN", "COPY", "ADD", "WORKDIR"}


def ler_dockerfile(texto: str) -> list[tuple[int, str, str]]:
    """Devolve [(linha, INSTRUÇÃO, argumento)].

    ⚠️ Junta continuações com `\\` — uma instrução pode ocupar 10 linhas
       e ainda ser UMA camada. Ignorar isso é o erro nº 1 de quem tenta
       analisar Dockerfile com um `split("\\n")` ingênuo.
    """
    passos, buffer, inicio = [], "", 0
    for n, bruta in enumerate(texto.splitlines(), 1):
        s = bruta.strip()
        if (not s or s.startswith("#")) and not buffer:
            continue
        if buffer:
            buffer += " " + s.rstrip("\\").strip()
        else:
            inicio, buffer = n, s.rstrip("\\").strip()
        if bruta.rstrip().endswith("\\"):
            continue
        if buffer:
            partes = buffer.split(None, 1)
            passos.append((inicio, partes[0].upper(),
                           partes[1] if len(partes) > 1 else ""))
        buffer = ""
    return passos


DOCKERFILE_INGENUO = """
FROM python:latest
RUN apt-get update
RUN apt-get install -y gcc libpq-dev curl
ENV ATLAS_SECRET_KEY=super-secreto-de-producao-123456
ARG DB_PASSWORD=senha-do-banco-aurora
WORKDIR /app
COPY . /app
RUN pip install -r requirements.txt
ADD dados/vendas.csv /app/dados/
EXPOSE 8000
CMD uvicorn atlas.api.aplicacao:criar_app --host 127.0.0.1
"""

passos = ler_dockerfile(DOCKERFILE_INGENUO)
camadas = [p for p in passos if p[1] in INSTRUCOES_QUE_CRIAM_CAMADA]

print(f"{len(passos)} instruções → {len(camadas)} camadas\n")
for i, (n, instr, arg) in enumerate(passos, 1):
    marca = "▣" if instr in INSTRUCOES_QUE_CRIAM_CAMADA else "·"
    print(f"  {marca} L{n:<3} {instr:<10} {arg[:52]}")
print("\n  ▣ = cria camada    · = só metadado")

> 🔴 **A consequência que assusta: apagar não diminui.**
>
> ```dockerfile
> COPY segredos.json /app/          ← camada 3: o arquivo entra
> RUN rm /app/segredos.json         ← camada 4: marca como apagado
> ```
>
> A camada 4 registra "este arquivo não existe mais". **A camada 3 continua tendo o arquivo.** Quem baixar a imagem pode extrair a camada 3 e ler tudo.
>
> É exatamente assim que credenciais vazam em imagens públicas. `docker history` e `docker save` expõem cada camada.
>
> 🧭 **A regra:** o que não deve existir na imagem **nunca pode entrar**. Não adianta apagar depois.

## 2. 🎯 O cache — por que a ordem das linhas importa

O Docker reaproveita uma camada se **ela e todas as anteriores** tiverem a mesma entrada. Uma camada invalidada invalida **todas as seguintes**.

Vamos medir.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Simulador de cache
# ═══════════════════════════════════════════════════════════════
CUSTO_TIPICO = {"FROM": 0.0, "WORKDIR": 0.1, "COPY": 0.5, "ADD": 0.5}


def custo(instr: str, arg: str) -> float:
    """Segundos típicos de cada passo, quando NÃO vem do cache."""
    if instr != "RUN":
        return CUSTO_TIPICO.get(instr, 0.0)
    if "pip install" in arg or "poetry install" in arg:
        return 45.0
    if "apt-get" in arg:
        return 20.0
    if "npm" in arg or "yarn" in arg:
        return 30.0
    return 2.0


def fontes(arg: str) -> list[str]:
    """Os caminhos de origem de um COPY/ADD (tudo menos o destino)."""
    partes = [p for p in arg.split() if not p.startswith("--")]
    return partes[:-1] if len(partes) > 1 else partes


def simular_cache(passos, alterados: list[str]):
    """Quais camadas o Docker reaproveita, dados os arquivos alterados.

    🔑 A regra é simples e implacável: assim que UMA camada é
       invalidada, TODAS as seguintes também são — mesmo que os
       arquivos delas não tenham mudado.
    """
    resultado, invalidado = [], False
    for n, instr, arg in passos:
        if instr not in INSTRUCOES_QUE_CRIAM_CAMADA:
            continue
        bate = False
        if not invalidado and instr in ("COPY", "ADD"):
            for fonte in fontes(arg):
                fonte = fonte.rstrip("/")
                for alvo in alterados:
                    if fonte in (".", "./") or alvo == fonte \
                       or alvo.startswith(fonte + "/") or fnmatch.fnmatch(alvo, fonte):
                        bate = True
                        break
                if bate:
                    break
        if bate:
            invalidado = True
        resultado.append((n, instr, arg, not invalidado,
                          0.0 if not invalidado else custo(instr, arg)))
    return resultado, sum(r[4] for r in resultado)


def relatorio_cache(rotulo, passos, alterados):
    res, total = simular_cache(passos, alterados)
    print(f"── {rotulo} ──")
    for n, instr, arg, veio_do_cache, seg in res:
        marca = "CACHE  " if veio_do_cache else "REBUILD"
        print(f"   {marca} {instr:<8} {arg[:36]:<38} {seg:5.1f}s")
    print(f"   {'':<8} {'TOTAL':<8} {'':<38} {total:5.1f}s\n")
    return total


print("Cenário: você mudou UMA linha em src/atlas/api/rotas/produtos.py\n")
ALTERADO = ["src/atlas/api/rotas/produtos.py"]
t_ruim = relatorio_cache("ORDEM INGÊNUA (COPY . antes das dependências)",
                         passos, ALTERADO)

In [ ]:
ORDEM_BOA = """
FROM python:3.12-slim
WORKDIR /app
COPY pyproject.toml ./
RUN pip install --no-cache-dir .
COPY src/ ./src/
"""
t_bom = relatorio_cache("ORDEM CORRETA (dependências primeiro)",
                        ler_dockerfile(ORDEM_BOA), ALTERADO)

print(f"🎯 {t_ruim:.1f}s  →  {t_bom:.1f}s      ({t_ruim / max(t_bom, 0.1):.0f}× mais rápido)")
print("\n   A diferença é UMA linha trocada de lugar.")
print(f"   Em 30 builds por dia, isso é {(t_ruim - t_bom) * 30 / 60:.0f} minutos economizados —")
print("   e, o que importa mais, o ciclo de feedback deixa de doer.")

> 🎯 **A regra de ouro do Dockerfile: do que muda MENOS para o que muda MAIS.**
>
> ```dockerfile
> FROM python:3.12-slim        ← muda a cada meses
> RUN apt-get install ...      ← muda a cada meses
> COPY pyproject.toml ./       ← muda a cada semanas
> RUN pip install .            ← 🔑 45 segundos, protegidos
> COPY src/ ./src/             ← muda a cada minutos
> ```
>
> `pyproject.toml` é copiado **sozinho e antes** do código exatamente para que o `pip install` fique numa camada que o seu código não invalida.
>
> 💭 **É a mesma ideia do índice do M03:** você paga um custo de organização uma vez para que a operação frequente fique barata.

In [ ]:
# ⚠️ E se você mudar a dependência?
print("Cenário: você adicionou uma biblioteca ao pyproject.toml\n")
relatorio_cache("ORDEM CORRETA, mas mudou a dependência",
                ler_dockerfile(ORDEM_BOA), ["pyproject.toml"])

print("💭 Aqui o rebuild é INEVITÁVEL e correto: a dependência mudou,")
print("   o `pip install` precisa rodar. O cache não é mágica — ele")
print("   evita trabalho DESNECESSÁRIO, não trabalho necessário.")

## 3. `.dockerignore`

O `docker build` começa enviando o **contexto** — a pasta inteira — para o daemon. Sem `.dockerignore`, isso inclui `.venv/`, `.git/`, `__pycache__/` e os dados de teste.

In [ ]:
# Vamos medir num projeto de mentira, mas com tamanhos realistas
CONTEXTO = {
    "src/":            ("código-fonte",        180_000,  True),
    "tests/":          ("testes",               90_000,  False),
    "pyproject.toml":  ("dependências",          3_000,  True),
    ".venv/":          ("ambiente virtual",  310_000_000, False),
    ".git/":           ("histórico completo", 48_000_000, False),
    "__pycache__/":    ("bytecode",           2_400_000, False),
    "dados/brutos/":   ("CSVs de teste",     15_000_000, False),
    "saida/":          ("relatórios gerados", 8_000_000, False),
    ".env":            ("🔴 SEGREDOS",              600, False),
    "notebooks/":      ("notebooks do manual", 22_000_000, False),
}

total = sum(t for _, t, _ in CONTEXTO.values())
util = sum(t for _, t, u in CONTEXTO.values() if u)

print(f"{'CAMINHO':<18}{'O QUE É':<22}{'TAMANHO':>12}   VAI PARA A IMAGEM?")
print("─" * 74)
for caminho, (desc, tam, usar) in CONTEXTO.items():
    print(f"{caminho:<18}{desc:<22}{tam / 1e6:>9.1f} MB   {'✅ sim' if usar else '🔴 NÃO'}")

print("─" * 74)
print(f"{'TOTAL enviado':<40}{total / 1e6:>9.1f} MB")
print(f"{'Realmente necessário':<40}{util / 1e6:>9.1f} MB")
print(f"\n🔴 {(1 - util / total) * 100:.1f}% do contexto é desperdício —")
print("   e o `.env` estaria dentro da imagem.")

In [ ]:
DOCKERIGNORE = BASE / ".dockerignore"
DOCKERIGNORE.write_text("""# ═══════════════════════════════════════════════════════════
#  .dockerignore — o que NÃO entra no contexto do build
#
#  💭 Ele parece com o .gitignore, mas o objetivo é outro:
#     .gitignore   → o que não versionar
#     .dockerignore → o que não ENVIAR para o daemon
#
#  Um arquivo pode estar versionado e mesmo assim não ter nada
#  que fazer dentro da imagem (os testes, por exemplo).
# ═══════════════════════════════════════════════════════════

# 🔴 SEGREDOS — antes de tudo
.env
.env.*
!.env.example
*.pem
*.key
secrets/

# Ambiente virtual: é do SEU sistema operacional, não do container
.venv/
venv/
ENV/

# Histórico do Git: dezenas de MB que a imagem não usa
.git/
.gitignore
.github/

# Cache e artefatos do Python
__pycache__/
*.py[cod]
.pytest_cache/
.mypy_cache/
.ruff_cache/
.coverage
htmlcov/
*.egg-info/
dist/
build/

# Dados e saídas: vêm de volume, não da imagem
dados/brutos/
dados/processados/
saida/
*.db
*.db-wal
*.db-shm

# Desenvolvimento
.vscode/
.idea/
notebooks/
docs/
*.md
!README.md

# Docker (evita recursão)
Dockerfile*
docker-compose*.yml
.dockerignore
""", encoding="utf-8")

linhas = [l for l in DOCKERIGNORE.read_text(encoding="utf-8").splitlines()
          if l.strip() and not l.startswith("#")]
print(f"✅ .dockerignore com {len(linhas)} regras\n")
print(f"contexto antes : {total / 1e6:>8.1f} MB")
print(f"contexto depois: {util / 1e6:>8.1f} MB")
print(f"redução        : {(1 - util / total) * 100:>8.1f}%")

> 🔴 **A primeira linha do `.dockerignore` é `.env`.**
>
> Sem ela, um `COPY . /app` coloca as suas credenciais dentro da imagem — e a imagem vai para um registro, e o registro pode ser público, e a camada é extraível.
>
> ⚠️ **`.gitignore` não protege aqui.** São arquivos diferentes, com propósitos diferentes. Ter `.env` no `.gitignore` e não ter no `.dockerignore` é uma das formas mais comuns de vazar segredo.

## 4. Multi-stage — 1,4 GB vira 180 MB

Para instalar o `psycopg`, você precisa de `gcc` e `libpq-dev`. Para **rodar** o Atlas, não.

Multi-stage build resolve isso: compile num estágio, copie só o resultado para outro.

In [ ]:
# Estimativa de tamanho, camada a camada
def estimar(rotulo, componentes):
    total = sum(t for _, t in componentes)
    print(f"── {rotulo} ──")
    for nome, tam in componentes:
        barra = "█" * max(1, int(tam / 25))
        print(f"   {nome:<26}{tam:>5} MB  {barra}")
    print(f"   {'TOTAL':<26}{total:>5} MB\n")
    return total


t1 = estimar("UM ESTÁGIO SÓ (o ingênuo)", [
    ("python:3.12 (completo)", 1013),
    ("gcc + build-essential", 250),
    ("libpq-dev + headers", 40),
    ("cache do apt", 45),
    ("dependências Python", 180),
    ("cache do pip", 60),
    ("código do Atlas", 2),
])

t2 = estimar("MULTI-STAGE", [
    ("python:3.12-slim (base)", 130),
    ("libpq5 (só a runtime)", 8),
    ("dependências Python", 180),
    ("código do Atlas", 2),
])

print(f"🎯 {t1} MB  →  {t2} MB      ({t1 / t2:.1f}× menor)")
print(f"\n   Economia por download: {(t1 - t2) / 1024:.1f} GB")
print(f"   Em 50 deploys/mês: {(t1 - t2) * 50 / 1024:.0f} GB de rede")
print("\n💭 E o ganho maior nem é o disco — é a SUPERFÍCIE DE ATAQUE.")
print("   `gcc` dentro de um container de produção é uma ferramenta")
print("   pronta para quem conseguir entrar.")

In [ ]:
DOCKERFILE_BOM = BASE / "Dockerfile"
DOCKERFILE_BOM.write_text("""# ═══════════════════════════════════════════════════════════════
#  Atlas API — imagem de produção
#
#  Construir:  docker build -t atlas-api:1.2.0 .
#  Rodar:      docker run -p 8000:8000 --env-file .env atlas-api:1.2.0
# ═══════════════════════════════════════════════════════════════

# ───────────────────────── ESTÁGIO 1: construir ─────────────────
# 🔑 Aqui pode haver compilador, headers, ferramenta de build.
#    Nada disto vai para a imagem final.
FROM python:3.12-slim AS construtor

# ⚠️ update e install NA MESMA camada RUN.
#    Em RUNs separados, o Docker reaproveita o `update` antigo e o
#    `install` usa uma lista de pacotes desatualizada — o famoso
#    "package not found" que some quando você limpa o cache.
RUN apt-get update && apt-get install -y --no-install-recommends \\
        gcc \\
        libpq-dev \\
    && rm -rf /var/lib/apt/lists/*
#      ↑ limpar na MESMA camada; num RUN seguinte não adiantaria nada

WORKDIR /app

# 🎯 As dependências vêm ANTES do código. É a linha mais importante
#    do arquivo: ela protege o `pip install` do cache invalidado.
COPY pyproject.toml ./
RUN pip install --no-cache-dir --prefix=/instalado .

# ───────────────────────── ESTÁGIO 2: rodar ─────────────────────
FROM python:3.12-slim

# Só a biblioteca de runtime do Postgres — sem headers, sem gcc
RUN apt-get update && apt-get install -y --no-install-recommends \\
        libpq5 \\
    && rm -rf /var/lib/apt/lists/*

# 🔴 Usuário sem privilégio. Se alguém escapar da aplicação, escapa
#    para um usuário que não pode nada.
RUN useradd --create-home --uid 1000 atlas

WORKDIR /app

# 🔑 Só o resultado do estágio anterior. O gcc ficou para trás.
COPY --from=construtor /instalado /usr/local
COPY --chown=atlas:atlas src/ ./src/

USER atlas

ENV PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONPATH=/app/src
#   ↑ PYTHONUNBUFFERED=1 é OBRIGATÓRIO: sem ele o Python bufferiza o
#     stdout e os seus logs só aparecem quando o buffer enche — ou
#     nunca, se o container morrer antes.

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \\
    CMD python -c "import httpx,sys; sys.exit(0 if httpx.get('http://127.0.0.1:8000/saude').status_code==200 else 1)"

# 🔴 Forma JSON (exec), não shell. Na forma shell, o processo real
#    vira filho de /bin/sh e NÃO recebe o SIGTERM do `docker stop`.
ENTRYPOINT ["python", "-m", "uvicorn"]
CMD ["atlas.api.aplicacao:criar_app", "--factory", \\
     "--host", "0.0.0.0", "--port", "8000"]
#            ↑ 0.0.0.0, nunca 127.0.0.1
""", encoding="utf-8")

print(f"✅ Dockerfile criado ({len(DOCKERFILE_BOM.read_text(encoding='utf-8').splitlines())} linhas)")
bom = ler_dockerfile(DOCKERFILE_BOM.read_text(encoding="utf-8"))
print(f"   {len(bom)} instruções → "
      f"{len([p for p in bom if p[1] in INSTRUCOES_QUE_CRIAM_CAMADA])} camadas")

## 5. Escolhendo a base

In [ ]:
bases = [
    ["python:3.12",          1013, "completo",   "compilar, depurar, desenvolvimento"],
    ["python:3.12-slim",      130, "enxuto",     "✅ o padrão sensato"],
    ["python:3.12-alpine",     51, "musl libc",  "⚠️ cuidado — veja abaixo"],
    ["gcr.io/distroless/python3", 52, "sem shell", "produção endurecida"],
]
tabela(["IMAGEM", "MB", "O QUE É", "QUANDO USAR"],
       bases, [30, 6, 12, 34])

print("""
🔴 A ARMADILHA DO ALPINE COM PYTHON

   Alpine usa `musl` no lugar da `glibc`. As "wheels" pré-compiladas do
   PyPI são feitas para glibc — então no Alpine o pip COMPILA tudo do
   zero.

   Resultado prático:
     · build 5 a 10× mais lento
     · precisa de gcc na imagem (que você queria evitar)
     · a imagem final às vezes fica MAIOR que a slim
     · bugs sutis de biblioteca C e diferença de DNS

   🧭 Alpine é excelente para Go e para binários estáticos.
      Para Python, comece com `slim`. Só vá para Alpine se medir
      um ganho de verdade.

⚠️ E DISTROLESS: não tem shell. `docker exec -it ... sh` não funciona.
   Ótimo para segurança, doloroso para depurar. Deixe para quando a
   imagem já estiver estável.
""")

## 6. 🔴 Segredos — por que `ARG` não esconde nada

In [ ]:
print("""
🔴 AS TRÊS FORMAS ERRADAS

   1. ENV ATLAS_SECRET_KEY=abc123
      Fica na imagem para sempre. `docker inspect` mostra.

   2. ARG DB_PASSWORD=senha
      "Mas ARG é só do build!" — sim, e mesmo assim aparece em
      `docker history`. Qualquer um que baixe a imagem lê.

   3. COPY .env /app/.env
      Entra numa camada. `RUN rm .env` depois NÃO adianta:
      a camada anterior continua tendo o arquivo.
""")

print("✅ AS FORMAS CERTAS\n")
print("   Em tempo de EXECUÇÃO (o caso comum):")
print("      docker run --env-file .env atlas-api")
print("      docker run -e ATLAS_SECRET_KEY=\"$ATLAS_SECRET_KEY\" atlas-api")
print("      (e, em orquestrador, os secrets do Kubernetes/Swarm)\n")
print("   Em tempo de BUILD (token privado do pip, por exemplo):")
print("      RUN --mount=type=secret,id=pip_token \\")
print("          pip install --index-url \"$(cat /run/secrets/pip_token)\" .")
print("      docker build --secret id=pip_token,src=./token.txt .")
print("\n   🔑 O `--mount=type=secret` monta o segredo SÓ durante aquele")
print("      RUN. Ele não vira camada e não aparece no histórico.")

In [ ]:
# Provando: o que `docker history` revelaria
docker("history atlas-api:ingenua --no-trunc", esperado="""
IMAGE          CREATED BY                                       SIZE
3f2a1b8c9d0e   CMD ["uvicorn" "atlas..."]                       0B
<missing>      EXPOSE 8000                                      0B
<missing>      ARG DB_PASSWORD=senha-do-banco-aurora            0B      ← 🔴
<missing>      ENV ATLAS_SECRET_KEY=super-secreto-de-produc...  0B      ← 🔴
<missing>      RUN pip install -r requirements.txt              187MB
<missing>      COPY . /app                                      412MB   ← 🔴 .env aqui dentro
""")

print("\n🔴 Três vazamentos numa imagem só — e ela 'funciona'.")
print("   Nenhum teste pega isso. Nenhum erro aparece. A imagem sobe")
print("   para o registro e o segredo vai junto.")

## 7. 🔴 Usuário não-root e o `SIGTERM` que não chega

In [ ]:
print("""
🔴 POR PADRÃO, O PROCESSO DENTRO DO CONTAINER É ROOT.

   "Mas é root só dentro do container." — depende:

   · Docker padrão (daemon como root): se o processo escapar do
     container, ele escapa COMO ROOT no host.
   · Volume montado (`-v /dados:/app/dados`): o processo escreve no
     host como root, e você fica com arquivos que não consegue apagar.
   · Rootless / user namespace: aí sim o root é mapeado.

   ✅ A correção é de três linhas:

      RUN useradd --create-home --uid 1000 atlas
      COPY --chown=atlas:atlas src/ ./src/
      USER atlas

   ⚠️ Depois do `USER`, você não instala mais nada. Ponha o `USER`
      DEPOIS de todos os `apt-get` e `pip install`.
""")

print("💭 E o UID 1000 não é aleatório: é o primeiro usuário comum no")
print("   Linux. Casar o UID de dentro com o de fora evita problema de")
print("   permissão em bind mount.")

In [ ]:
# CMD e ENTRYPOINT — as duas formas, e por que uma delas quebra
print("""
FORMA SHELL                       FORMA EXEC (JSON)
CMD uvicorn app --host 0.0.0.0    CMD ["uvicorn","app","--host","0.0.0.0"]

vira:  /bin/sh -c "uvicorn ..."   vira:  uvicorn ...
       PID 1 = sh                        PID 1 = uvicorn
       uvicorn = filho do sh

🔴 O PROBLEMA:

   `docker stop` manda SIGTERM para o PID 1.
   Na forma shell, o PID 1 é o `sh` — que NÃO repassa o sinal.

   Resultado: o uvicorn nunca sabe que deve encerrar. O Docker espera
   10 segundos, desiste e manda SIGKILL.

   Consequência prática: conexões cortadas no meio, transações
   abandonadas, e todo `docker stop` demorando 10 segundos.
""")

print("✅ Use SEMPRE a forma JSON.\n")
print("💡 ENTRYPOINT × CMD:")
print("   ENTRYPOINT  o que SEMPRE roda        (o executável)")
print("   CMD         argumentos PADRÃO        (substituíveis)")
print()
print('   ENTRYPOINT ["python", "-m", "uvicorn"]')
print('   CMD ["app:criar_app", "--host", "0.0.0.0"]')
print()
print("   docker run atlas-api                → usa o CMD")
print("   docker run atlas-api app:outra      → substitui só o CMD")
print("   docker run --entrypoint sh -it ...  → 🔧 para depurar")

## 8. Volumes vs bind mounts

In [ ]:
comparacao = [
    ["Quem gerencia",    "o Docker",              "você"],
    ["Onde fica",        "/var/lib/docker/volumes", "onde você disser"],
    ["Sintaxe",          "-v nome:/caminho",      "-v /host/dir:/caminho"],
    ["Desempenho",       "nativo",                "⚠️ lento no Win/macOS"],
    ["Backup",           "docker volume",         "cp comum"],
    ["Sobrevive ao rm",  "✅ sim",                "✅ sim"],
    ["Use para",         "dados do banco",        "código em desenvolvimento"],
]
tabela(["", "VOLUME", "BIND MOUNT"], comparacao, [18, 26, 28])

print()
docker("run -d -v atlas-dados:/var/lib/postgresql/data postgres:16",
       esperado="# ✅ produção: o Docker cuida do armazenamento")
print()
docker("run -v $(pwd)/src:/app/src atlas-api:dev",
       esperado="# ✅ desenvolvimento: editou no host, mudou no container")
print()
docker("run --tmpfs /tmp:size=64m atlas-api",
       esperado="# ✅ só na memória: some ao parar, nunca toca o disco")

print("\n💡 O bind mount de código + `--reload` do uvicorn dá o mesmo")
print("   ciclo de desenvolvimento de sempre, com o ambiente do container.")
print("\n⚠️ Mas NÃO use bind mount de código em produção: a imagem deixa")
print("   de ser a fonte da verdade, e o que roda passa a depender do")
print("   que está no disco daquela máquina.")

## 🔧 Prática guiada — o analisador de Dockerfile

Agora juntamos tudo num verificador que roda **antes** do `docker build`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Analisador de Dockerfile
# ═══════════════════════════════════════════════════════════════
SEGREDO = re.compile(
    r"(SECRET|PASSWORD|SENHA|TOKEN|API_?KEY|PRIVATE_KEY|CREDENTIAL)"
    r"\w*\s*[= ]\s*[\"']?[^\s\"']{6,}", re.I)


def analisar(texto: str):
    """Devolve (passos, achados). Achado = (gravidade, linha, o quê, dica)."""
    passos = ler_dockerfile(texto)
    achados: list[tuple[str, int, str, str]] = []

    def add(g, n, msg, dica=""):
        achados.append((g, n, msg, dica))

    froms = [p for p in passos if p[1] == "FROM"]
    tem_user = any(p[1] == "USER" for p in passos)

    for n, instr, arg in passos:
        if instr == "FROM":
            img = re.split(r"\s+as\s+", arg, flags=re.I)[0].strip()
            if "@sha256:" not in img:
                if ":" not in img.split("/")[-1]:
                    add("🔴", n, f"FROM sem tag: {img}", "fixe a versão")
                elif img.endswith(":latest"):
                    add("🔴", n, f"FROM :latest — {img}",
                        "'latest' não é 'a mais nova'; fixe a versão")
        if instr == "RUN":
            if "apt-get update" in arg and "apt-get install" not in arg:
                add("🔴", n, "apt-get update num RUN separado",
                    "o cache reaproveita a lista velha; junte update+install")
            if "apt-get install" in arg and "--no-install-recommends" not in arg:
                add("⚠️ ", n, "apt-get install sem --no-install-recommends",
                    "traz dezenas de MB que você não pediu")
            if "apt-get install" in arg and "rm -rf /var/lib/apt/lists" not in arg:
                add("⚠️ ", n, "apt sem limpar /var/lib/apt/lists",
                    "limpe NA MESMA camada — depois não adianta")
            if re.search(r"\bpip install\b", arg) and "--no-cache-dir" not in arg:
                add("⚠️ ", n, "pip install sem --no-cache-dir",
                    "o cache do pip fica na imagem")
            if SEGREDO.search(arg):
                add("🔴", n, "possível SEGREDO num RUN",
                    "fica no histórico da imagem para sempre")
            if arg.strip().startswith("sudo"):
                add("⚠️ ", n, "sudo no Dockerfile", "você já é root no build")
        if instr == "ENV" and SEGREDO.search(arg):
            add("🔴", n, "SEGREDO em ENV", "docker inspect revela")
        if instr == "ARG" and SEGREDO.search(arg):
            add("🔴", n, "SEGREDO em ARG", "docker history revela")
        if instr == "ADD" and not arg.startswith(("http://", "https://")):
            add("⚠️ ", n, "ADD com arquivo local",
                "use COPY; ADD também extrai tar e baixa URL")
        if instr in ("CMD", "ENTRYPOINT"):
            if not arg.strip().startswith("["):
                add("🔴", n, f"{instr} em forma de shell",
                    "use JSON: na forma shell o SIGTERM não chega ao processo")
            if "127.0.0.1" in arg or "localhost" in arg:
                add("🔴", n, "escuta em 127.0.0.1",
                    "use 0.0.0.0 — o -p não alcança o loopback do container")

    if not tem_user:
        add("🔴", 0, "nenhum USER: o processo roda como root",
            "crie um usuário e use USER antes do CMD")
    if not any(p[1] in ("CMD", "ENTRYPOINT") for p in passos):
        add("🔴", 0, "nem CMD nem ENTRYPOINT",
            "o container não sabe o que executar")
    if len(froms) == 1 and any(
            ("gcc" in a or "build-essential" in a) for _, i, a in passos if i == "RUN"):
        add("⚠️ ", 0, "compilador na imagem final",
            "use multi-stage: compile num estágio, copie o resultado")

    # ordem do cache
    i_dep = next((k for k, (n, i, a) in enumerate(passos)
                  if i in ("COPY", "ADD")
                  and re.search(r"requirements|pyproject|poetry\.lock|package\.json", a)),
                 None)
    i_tudo = next((k for k, (n, i, a) in enumerate(passos)
                   if i == "COPY" and re.match(r"^\.?\s+", a)), None)
    if i_dep is not None and i_tudo is not None and i_tudo < i_dep:
        add("🔴", passos[i_tudo][0], "COPY . antes das dependências",
            "inverta: qualquer mudança de código invalida o pip install")
    elif i_dep is None and i_tudo is not None:
        add("⚠️ ", passos[i_tudo][0], "só há COPY . — sem etapa de dependências",
            "copie pyproject/requirements primeiro")

    return passos, achados


def relatorio(nome: str, texto: str) -> int:
    passos, achados = analisar(texto)
    camadas_ = [p for p in passos if p[1] in INSTRUCOES_QUE_CRIAM_CAMADA]
    graves = sum(1 for g, *_ in achados if g == "🔴")
    print(f"╔{'═' * 66}╗")
    print(f"║ {nome:<64} ║")
    print(f"╚{'═' * 66}╝")
    print(f"  {len(passos)} instruções · {len(camadas_)} camadas · "
          f"{len(achados)} achados ({graves} graves)\n")
    for g, n, msg, dica in achados:
        print(f"  {g} {('L' + str(n)) if n else '  ':<5} {msg}")
        if dica:
            print(f"        └─ {dica}")
    if not achados:
        print("  ✅ nenhum problema encontrado")
    print()
    return graves


graves_ruim = relatorio("Dockerfile INGÊNUO", DOCKERFILE_INGENUO)

In [ ]:
graves_bom = relatorio("Dockerfile do Atlas (multi-stage)",
                       DOCKERFILE_BOM.read_text(encoding="utf-8"))

print(f"🎯 {graves_ruim} problemas graves  →  {graves_bom}")
print("\n💭 Repare que NENHUM desses problemas impediria o build.")
print("   Os dois Dockerfiles constroem e rodam. A diferença aparece")
print("   depois: no tamanho, no tempo de build, e na auditoria de")
print("   segurança — quando já está em produção.")

In [ ]:
# Um verificador para o CI (M09)
def verificar_para_ci(caminho: Path) -> int:
    """Devolve 0 se passou, 1 se há problema grave.

    💡 É assim que isto vira um portão no CI: o pipeline chama, olha o
       código de saída, e barra o merge se houver 🔴.
    """
    _, achados = analisar(caminho.read_text(encoding="utf-8"))
    graves = [a for a in achados if a[0] == "🔴"]
    if graves:
        print(f"🔴 {caminho.name}: {len(graves)} problema(s) grave(s)")
        for _, n, msg, _ in graves:
            print(f"   L{n}: {msg}")
        return 1
    print(f"✅ {caminho.name}: aprovado")
    return 0


print("Rodando como o CI rodaria:\n")
codigo = verificar_para_ci(DOCKERFILE_BOM)
print(f"\n[código de saída: {codigo}]")
print("\n💡 Em produção use também o `hadolint`, que é o linter de")
print("   Dockerfile consagrado. O valor de escrever o seu uma vez é")
print("   entender O QUE ele verifica — e por quê.")

In [ ]:
print("Estrutura criada nesta aula:\n")
arvore(BASE)

## 📝 Exercícios

**E1.** Conte as camadas de um Dockerfile seu à mão, depois confira com o `ler_dockerfile()`. Explique por que `ENV` não conta.

**E2.** 🔴 Demonstre que apagar não diminui: `COPY` um arquivo de 10 MB e `RUN rm` na linha seguinte. Explique por que a imagem não encolhe.

**E3.** Use o `simular_cache()` com três cenários: mudou só o código, mudou só a dependência, mudou o `Dockerfile`. Compare os tempos.

**E4.** Reordene um Dockerfile ruim para maximizar o cache e meça o ganho com o simulador.

**E5.** Escreva um `.dockerignore` para o seu `projeto_Atlas` e calcule a redução do contexto.

**E6.** 🔴 Prove que `.gitignore` não substitui `.dockerignore`: monte um caso em que `.env` está no primeiro e entra na imagem mesmo assim.

**E7.** Converta um Dockerfile de estágio único em multi-stage. Compare os tamanhos estimados.

**E8.** Compare `python:3.12`, `slim` e `alpine` para o Atlas. Justifique a escolha por escrito.

**E9.** 🔴 Encontre os três vazamentos de segredo do `DOCKERFILE_INGENUO` e escreva a correção de cada um.

**E10.** Adicione usuário não-root a um Dockerfile. Explique por que o `USER` vai depois dos `RUN` de instalação.

**E11.** 🔴 Demonstre a diferença entre `CMD` shell e JSON num script que captura `SIGTERM`. Meça o tempo de `docker stop` nos dois casos.

**E12.** Escreva um `HEALTHCHECK` para a API do Atlas e explique cada parâmetro (`interval`, `timeout`, `start-period`, `retries`).

**E13.** Explique, num comentário, quando usar volume e quando usar bind mount — com um exemplo de cada no Atlas.

**E14.** Estenda o `analisar()` com três verificações novas. Sugestões: `WORKDIR` com caminho relativo, `EXPOSE` de porta privilegiada, ausência de `HEALTHCHECK`.

**E15.** 🔴 Rode o `verificar_para_ci()` contra o Dockerfile do seu `projeto_Atlas` e conserte tudo até ele sair com código 0.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```dockerfile
# ═══ A ORDEM: do que muda MENOS para o que muda MAIS 🎯 ═══
FROM python:3.12-slim AS construtor      # 🔴 fixe a versão, nunca :latest
RUN apt-get update && apt-get install -y --no-install-recommends gcc \
    && rm -rf /var/lib/apt/lists/*       # update+install+limpeza NA MESMA camada
WORKDIR /app
COPY pyproject.toml ./                   # 🔑 dependências ANTES do código
RUN pip install --no-cache-dir --prefix=/instalado .

FROM python:3.12-slim                    # ═══ estágio final ═══
RUN useradd --create-home --uid 1000 atlas
WORKDIR /app
COPY --from=construtor /instalado /usr/local
COPY --chown=atlas:atlas src/ ./src/     # o código por último
USER atlas                               # 🔴 depois de tudo que instala
ENV PYTHONUNBUFFERED=1                   # 🔴 senão o log não aparece
EXPOSE 8000
HEALTHCHECK --interval=30s --start-period=10s CMD [...]
ENTRYPOINT ["python", "-m", "uvicorn"]   # 🔴 JSON, não shell
CMD ["app:criar_app", "--host", "0.0.0.0"]   # 🔴 0.0.0.0, não 127.0.0.1
```

```bash
# ═══ Camadas ═══
# CRIAM: FROM RUN COPY ADD WORKDIR
# NÃO:   ENV ARG EXPOSE LABEL USER CMD ENTRYPOINT VOLUME HEALTHCHECK
# 🔴 apagar NÃO diminui — a camada anterior guarda o arquivo

# ═══ Cache ═══
# uma camada invalidada invalida TODAS as seguintes
# COPY deps → RUN install → COPY código   ← a ordem que protege o install

# ═══ .dockerignore ═══
# .env  .venv/  .git/  __pycache__/  tests/  notebooks/  saida/
# 🔴 a primeira linha é .env

# ═══ Segredos ═══
# 🔴 ENV / ARG / COPY .env      → ficam na imagem
# ✅ docker run --env-file .env  (execução)
# ✅ RUN --mount=type=secret,id=x  (build)

# ═══ Construir ═══
docker build -t atlas-api:1.2.0 .
docker build --target construtor .     # para só no estágio X
docker build --no-cache .              # ignora o cache
docker build --progress=plain .        # mostra a saída de cada RUN
docker history atlas-api:1.2.0         # 🔍 camada a camada
docker image ls                        # tamanhos

# ═══ Volumes ═══
-v nome:/caminho              volume    → dados do banco
-v $(pwd)/src:/app/src        bind      → código em desenvolvimento
--tmpfs /tmp:size=64m         memória   → temporário, nunca toca o disco
```

## ✅ Checklist de saída

**Camadas e cache**

- [ ] Sei quais instruções criam camada
- [ ] 🔴 **Sei que apagar um arquivo não o remove da imagem**
- [ ] Ordeno o Dockerfile do que muda menos para o que muda mais
- [ ] Copio as dependências **antes** do código
- [ ] Junto `apt-get update` e `install` no mesmo `RUN`
- [ ] Limpo `/var/lib/apt/lists` na mesma camada

**Tamanho**

- [ ] Uso multi-stage quando preciso compilar
- [ ] Escolho `slim` como padrão e sei por que Alpine é armadilha no Python
- [ ] Tenho `.dockerignore`, e a primeira linha é `.env`
- [ ] Uso `--no-cache-dir` no pip

**Segurança**

- [ ] 🔴 **Nenhum segredo em `ENV`, `ARG` ou `COPY`**
- [ ] Sei que `docker history` revela `ARG`
- [ ] 🔴 **Crio usuário e uso `USER`** antes do `CMD`
- [ ] Não deixo compilador na imagem final

**Execução**

- [ ] 🔴 **`CMD`/`ENTRYPOINT` em forma JSON**
- [ ] Sei por que o `SIGTERM` não chega na forma shell
- [ ] 🔴 **Escuto em `0.0.0.0`**
- [ ] Uso `PYTHONUNBUFFERED=1`
- [ ] Tenho `HEALTHCHECK`
- [ ] Sei a diferença entre `ENTRYPOINT` e `CMD`

**Dados**

- [ ] Sei quando usar volume e quando usar bind mount
- [ ] Não uso bind mount de código em produção

---

### ➡️ Próxima aula

**`08_03_Orquestracao.ipynb`** — Um container é fácil. Quatro conversando entre si, na ordem certa, é outro problema: redes, `docker compose`, `healthcheck` e por que `depends_on` não faz o que você acha.